In [2]:
import os
import re
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix, save_npz
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans

file_path = r"C:\Users\user\PycharmProjects\JupyterProject1\books\GoodReads_100k_books.csv"
data = pd.read_csv(file_path)

print("Original shape:", data.shape)
print(data.columns.tolist())

Original shape: (100000, 13)
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings']


In [4]:
editions = data.copy()

editions["bookformat_clean"] = (
    editions["bookformat"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

excluded_formats = [
    "kindle",
    "ebook",
    "e-book",
    "audiobook",
    "audio book",
    "audible",
    "audio cd",
    "mp3 cd"
]

format_pattern = "|".join(
    re.escape(x) for x in excluded_formats
)

editions["is_excluded_format"] = (
    editions["bookformat_clean"]
    .str.contains(
        format_pattern,
        case=False,
        na=False
    )
)

editions = editions[
    ~editions["is_excluded_format"]
].copy()

print(
    "Records after format filtering:",
    len(editions)
)

Records after format filtering: 89649


In [5]:
def normalize_title(title):

    if pd.isna(title):
        return ""

    title = str(title).lower().strip()

    # Remove edition information at the end
    title = re.sub(
        r"\s*\((?:"
        r"kindle edition|kindle|ebook|e-book|"
        r"audiobook|audio book|audio|"
        r"paperback|hardcover|"
        r"mass market paperback|"
        r"illustrated edition|"
        r"anniversary edition|edition"
        r")\)\s*$",
        "",
        title,
        flags=re.IGNORECASE
    )

    title = re.sub(
        r"[^a-z0-9\s]",
        " ",
        title
    )

    title = re.sub(
        r"\s+",
        " ",
        title
    )

    return title.strip()


def normalize_author(author):

    if pd.isna(author):
        return ""

    author = str(author).lower().strip()

    author = re.sub(
        r"[^a-z0-9\s]",
        " ",
        author
    )

    author = re.sub(
        r"\s+",
        " ",
        author
    )

    return author.strip()


editions["title_normalized"] = (
    editions["title"]
    .apply(normalize_title)
)

editions["author_normalized"] = (
    editions["author"]
    .apply(normalize_author)
)

In [6]:
editions["work_key"] = (
    editions["title_normalized"]
    + "||"
    + editions["author_normalized"]
)

print(
    "Unique title/author works:",
    editions["work_key"].nunique()
)

Unique title/author works: 88405


In [7]:
def clean_genres(value):

    if pd.isna(value):
        return []

    genres = [
        g.strip()
        for g in str(value).split(",")
        if g.strip()
    ]

    return list(dict.fromkeys(genres))


def combine_genres(series):

    combined = []

    for genre_list in series:

        for genre in genre_list:

            if genre not in combined:
                combined.append(genre)

    return combined


def first_non_null(series):

    values = series.dropna()

    if len(values) == 0:
        return np.nan

    return values.iloc[0]


editions["genre_list"] = (
    editions["genre"]
    .apply(clean_genres)
)

editions["pages_clean"] = pd.to_numeric(
    editions["pages"],
    errors="coerce"
)

editions.loc[
    editions["pages_clean"] == 0,
    "pages_clean"
] = np.nan


editions["rating_clean"] = pd.to_numeric(
    editions["rating"],
    errors="coerce"
)

editions.loc[
    editions["rating_clean"] == 0,
    "rating_clean"
] = np.nan


editions["totalratings"] = pd.to_numeric(
    editions["totalratings"],
    errors="coerce"
).fillna(0)

In [8]:
books = (
    editions
    .groupby("work_key", as_index=False)
    .agg(
        title=("title", first_non_null),
        author=("author", first_non_null),

        title_normalized=(
            "title_normalized",
            first_non_null
        ),

        author_normalized=(
            "author_normalized",
            first_non_null
        ),

        genre_list=(
            "genre_list",
            combine_genres
        ),

        description=(
            "desc",
            first_non_null
        ),

        pages_clean=(
            "pages_clean",
            "max"
        ),

        rating_clean=(
            "rating_clean",
            "mean"
        ),

        totalratings=(
            "totalratings",
            "max"
        )
    )
)

# Permanent internal ID
books["book_id"] = np.arange(
    len(books),
    dtype=np.int64
)

books["log_totalratings"] = np.log1p(
    books["totalratings"]
)

books = books.set_index(
    "book_id",
    drop=False
)

print(
    "Unique books:",
    len(books)
)

print(
    books[
        [
            "book_id",
            "title",
            "author"
        ]
    ].head()
)

Unique books: 88405
         book_id                                              title  \
book_id                                                               
0              0  è­¦è¦–åºç‰¹çŠ¯èª²007 [Keishichou Tokuhanka 00...   
1              1  ãƒ‡ãƒ¼ãƒˆãƒ»ã‚¢ãƒ»ãƒ©ã‚¤ãƒ– 02 - å››ç³¸ä¹ƒãƒ‘ã...   
2              2  ãƒ‡ãƒ¼ãƒˆãƒ»ã‚¢ãƒ»ãƒ©ã‚¤ãƒ– 03 - ç‹‚ä¸‰ã‚­ãƒ©ã...   
3              3  ã‚¢ã‚¯ã‚»ãƒ«ãƒ»ãƒ¯ãƒ¼ãƒ«ãƒ‰ 04 è’¼ç©ºã¸ã®é£›...   
4              4                              04 Colours And Shapes   

                                                author  
book_id                                                 
0                                 Eiri Kaji,èˆµ è‹±é‡Œ  
1        Koushi Tachibana,æ©˜ å…¬å¸,Tsunako,ã¤ãªã“  
2        Koushi Tachibana,æ©˜ å…¬å¸,Tsunako,ã¤ãªã“  
3                     Reki Kawahara,Reki Kawahara,HiMA  
4                         Lynne Bradbury,Lynn N Grundy  


In [9]:
mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(
    books["genre_list"]
).astype(np.float32)

genre_matrix = csr_matrix(
    genre_matrix
)

print(
    "Genre features:",
    genre_matrix.shape
)

Genre features: (88405, 1162)


In [11]:
numeric_features = books[
    [
        "log_totalratings",
        "rating_clean",
        "pages_clean"
    ]
].copy()

numeric_features = numeric_features.fillna(
    numeric_features.median()
)

scaler = StandardScaler()

numeric_matrix = scaler.fit_transform(
    numeric_features
).astype(np.float32)

# Weights
GENRE_WEIGHT = 0.7
POPULARITY_WEIGHT = 0.3
RATING_WEIGHT = 0.2
PAGES_WEIGHT = 0.1

genre_matrix = (
    genre_matrix * GENRE_WEIGHT
)

numeric_matrix[:, 0] *= (
    POPULARITY_WEIGHT
)

numeric_matrix[:, 1] *= (
    RATING_WEIGHT
)

numeric_matrix[:, 2] *= (
    PAGES_WEIGHT
)

numeric_matrix = csr_matrix(
    numeric_matrix
)

content_matrix = hstack(
    [
        genre_matrix,
        numeric_matrix
    ]
).tocsr()

print(
    "Final matrix:",
    content_matrix.shape
)

Final matrix: (88405, 1165)


In [12]:
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1
)

model.fit(content_matrix)

print(
    "NearestNeighbors trained."
)

NearestNeighbors trained.


In [13]:
N_CLUSTERS = 10

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=10
)

kmeans.fit(content_matrix)

books["cluster"] = (
    kmeans.labels_
)

print(
    "KMeans trained."
)

KMeans trained.


 book_id                                                                         title                                        author
   33424                                                                       Inferno                                     Dan Brown
   33425                                                                       Inferno             Dante Alighieri,Anthony M. Esolen
   33429                                                                       Inferno                              Sherrilyn Kenyon
   33428                                                                       Inferno                   Larry Niven,Jerry Pournelle
   33423                                                                       Inferno               August Strindberg,Carlo Picchio
   33427                                                                       Inferno Judith Reeves-Stevens,Garfield Reeves-Stevens
   33426                                                             

In [25]:
def search_books(query, limit=30):

    query = str(query).strip().lower()

    title = (
        books["title"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    author = (
        books["author"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    exact_title = title == query

    partial_title = title.str.contains(
        query,
        regex=False
    )

    partial_author = author.str.contains(
        query,
        regex=False
    )

    results = books[
        exact_title
        | partial_title
        | partial_author
    ].copy()

    results["_priority"] = 0

    results.loc[
        partial_author,
        "_priority"
    ] = 1

    results.loc[
        partial_title,
        "_priority"
    ] = 2

    results.loc[
        exact_title,
        "_priority"
    ] = 3

    results = results.sort_values(
        [
            "_priority",
            "totalratings"
        ],
        ascending=[
            False,
            False
        ]
    )

    return results.head(limit)

In [26]:
def recommend_books(book_id, n=10):

    # Find selected book
    selected = books[
        books["book_id"] == book_id
    ].iloc[0]

    selected_cluster = selected["cluster"]

    # Find books belonging to same cluster
    cluster_books = books[
        books["cluster"] == selected_cluster
    ]

    # Get their original matrix positions
    positions = cluster_books.index

    selected_position = books.index.get_loc(
        book_id
    )

    cluster_matrix = content_matrix[
        positions
    ]

    # Find nearest books within cluster
    distances, indices = model.kneighbors(
        content_matrix[selected_position],
        n_neighbors=min(
            n + 1,
            len(cluster_books)
        )
    )

    recommendations = books.iloc[
        indices[0]
    ].copy()

    recommendations["similarity"] = (
        1 - distances[0]
    )

    recommendations = recommendations[
        recommendations["book_id"] != book_id
    ]

    return recommendations.head(n)

In [30]:
recommend_books("Inferno")

IndexError: single positional indexer is out-of-bounds